# Dual-Hand Gesture Recognition - Complete Training Pipeline
## CNN-LSTM Model with MediaPipe Dual-Hand Landmarks

**Pipeline:** Preprocessing -> Augmentation -> Training -> Evaluation

| Step | Description | ~Time (GPU) |
|------|------------|-------------|
| 1 | Extract dual-hand MediaPipe landmarks | 5-10 min |
| 2 | Augment training data (5x) | 2-5 min |
| 3 | Train CNN-LSTM model (100 epochs) | 15-45 min |
| 4 | Evaluate + plots | 1-2 min |

## Step 0: Setup - Mount Drive & Install Dependencies

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# COLAB DEPENDENCIES (April 2026)
# Colab pre-installs: TensorFlow ~2.19, NumPy ~2.0, OpenCV,
#   scikit-learn, matplotlib, seaborn — DO NOT re-install those.
# Only install what's missing: mediapipe
# ============================================================
!pip install -q mediapipe==0.10.14

# Verify all packages
import tensorflow as tf
import numpy as np
import cv2
import mediapipe as mp
if not hasattr(mp, 'solutions'):
    print('ERROR: MediaPipe solutions missing! PLEASE RESTART RUNTIME (Runtime -> Restart session)')
import sklearn
import matplotlib
import seaborn
import sys

print(f'Python:      {sys.version.split()[0]}')
print(f'TensorFlow:  {tf.__version__}')
print(f'NumPy:       {np.__version__}')
print(f'OpenCV:      {cv2.__version__}')
print(f'MediaPipe:   {mp.__version__}')
print(f'Sklearn:     {sklearn.__version__}')
print(f'Matplotlib:  {matplotlib.__version__}')
print(f'Seaborn:     {seaborn.__version__}')
print()

# GPU check
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU available: {gpus[0]}')
    for gpu in gpus: tf.config.experimental.set_memory_growth(gpu, True)
else:
    print('WARNING: No GPU! Go to Runtime -> Change runtime type -> T4 GPU')

print('\nAll dependencies ready!')

## Step 0.1: Configuration

In [ ]:
import os, sys, json, pickle
# numpy, cv2, mediapipe, tf already imported in setup cell
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter

# ======================== PATHS ========================
DATASET_PATH = '/content/drive/MyDrive/dataset/dataset'
PREPROCESSED_DATA_PATH = '/content/preprocessed_data/'
MODEL_SAVE_PATH = '/content/models/'
EVALUATION_PATH = '/content/evaluation_results/'
AUGMENTED_DATA_PATH = '/content/augmented_data/'

# Also save to Drive for persistence
DRIVE_SAVE_PATH = '/content/drive/MyDrive/dataset/'

# ======================== MEDIAPIPE ========================
NUM_HANDS = 2
LANDMARKS_PER_HAND = 21
NUM_LANDMARKS = LANDMARKS_PER_HAND * NUM_HANDS  # 42
NUM_COORDS = 3
FEATURES_PER_HAND = LANDMARKS_PER_HAND * NUM_COORDS  # 63
NUM_FEATURES = NUM_LANDMARKS * NUM_COORDS  # 126

# ======================== SEQUENCE ========================
SEQUENCE_LENGTH = 30
MAX_HAND_DETECTION_FAILURES = 0.5

# ======================== MODEL ========================
CNN_FILTERS = [32, 64]
LSTM_UNITS = [64]
DENSE_UNITS = 128
DROPOUT_RATE = 0.5
LEARNING_RATE = 0.001

# ======================== TRAINING ========================
BATCH_SIZE = 16
EPOCHS = 150
VALIDATION_SPLIT = 0.2
TEST_SPLIT = 0.20
RANDOM_SEED = 42

# ======================== AUGMENTATION ========================
AUGMENTATION_FACTOR = 20
NOISE_STD = 0.02
SCALE_RANGE = (0.9, 1.1)
SHIFT_RANGE = 0.05
TEMPORAL_SHIFT = 3
ROTATION_ANGLE_RANGE = (-15, 15)

# Create directories
for p in [PREPROCESSED_DATA_PATH, MODEL_SAVE_PATH, EVALUATION_PATH, AUGMENTED_DATA_PATH]:
    os.makedirs(p, exist_ok=True)

# Verify dataset
if os.path.exists(DATASET_PATH):
    gesture_dirs = [d for d in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, d))]
    total_videos = sum(len([f for f in os.listdir(os.path.join(DATASET_PATH, d)) if f.endswith(('.mp4','.avi','.mov','.mkv'))]) for d in gesture_dirs)
    print(f'Dataset found: {len(gesture_dirs)} classes, ~{total_videos} videos')
else:
    print(f'Dataset NOT found at: {DATASET_PATH}')

print(f'TensorFlow: {tf.__version__}, NumPy: {np.__version__}')
print(f'Sequence: {SEQUENCE_LENGTH} frames x {NUM_FEATURES} features')

## Step 1: Data Preprocessing - Extract Dual-Hand MediaPipe Landmarks
Extracts hand landmarks from ALL gesture videos using BOTH hands.

In [ ]:
def initialize_mediapipe():
    mp_hands = mp.solutions.hands
    hands = mp_hands.Hands(static_image_mode=False, max_num_hands=2,
                           min_detection_confidence=0.5, min_tracking_confidence=0.5)
    return hands

def extract_landmarks_from_frame(frame, hands):
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(frame_rgb)
    if not results.multi_hand_landmarks:
        return None
    left_hand = np.zeros(FEATURES_PER_HAND, dtype=np.float32)
    right_hand = np.zeros(FEATURES_PER_HAND, dtype=np.float32)
    for idx, hand_landmarks in enumerate(results.multi_hand_landmarks):
        landmarks = []
        for lm in hand_landmarks.landmark:
            landmarks.extend([lm.x, lm.y, lm.z])
        landmarks = np.array(landmarks, dtype=np.float32)
        if results.multi_handedness and idx < len(results.multi_handedness):
            handedness = results.multi_handedness[idx].classification[0].label
        else:
            avg_x = np.mean([lm.x for lm in hand_landmarks.landmark])
            handedness = 'Left' if avg_x > 0.5 else 'Right'
        if handedness == 'Left':
            left_hand = landmarks
        else:
            right_hand = landmarks
    return np.concatenate([left_hand, right_hand])

def normalize_landmarks(landmarks_sequence):
    normalized = landmarks_sequence.copy()
    for i in range(len(normalized)):
        if np.all(normalized[i] == 0): continue
        for hand_idx in range(NUM_HANDS):
            start = hand_idx * FEATURES_PER_HAND
            end = start + FEATURES_PER_HAND
            hand_features = normalized[i, start:end]
            if np.all(hand_features == 0): continue
            frame_landmarks = hand_features.reshape(LANDMARKS_PER_HAND, NUM_COORDS)
            x_coords, y_coords, z_coords = frame_landmarks[:,0], frame_landmarks[:,1], frame_landmarks[:,2]
            x_range = x_coords.max() - x_coords.min()
            y_range = y_coords.max() - y_coords.min()
            if x_range > 0: frame_landmarks[:,0] = (x_coords - x_coords.min()) / x_range
            if y_range > 0: frame_landmarks[:,1] = (y_coords - y_coords.min()) / y_range
            z_max = np.abs(z_coords).max()
            if z_max > 0: frame_landmarks[:,2] = z_coords / z_max
            normalized[i, start:end] = frame_landmarks.flatten()
    return normalized

def extract_landmarks_from_video(video_path, hands):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): return None
    all_landmarks, total_frames, failed_frames = [], 0, 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        total_frames += 1
        landmarks = extract_landmarks_from_frame(frame, hands)
        if landmarks is not None:
            all_landmarks.append(landmarks)
        else:
            failed_frames += 1
            all_landmarks.append(np.zeros(NUM_FEATURES, dtype=np.float32))
    cap.release()
    if total_frames == 0: return None
    all_landmarks = np.array(all_landmarks)
    if len(all_landmarks) >= SEQUENCE_LENGTH:
        indices = np.linspace(0, len(all_landmarks)-1, SEQUENCE_LENGTH, dtype=int)
        sampled = all_landmarks[indices]
    else:
        padding_needed = SEQUENCE_LENGTH - len(all_landmarks)
        last_frame = all_landmarks[-1:] if len(all_landmarks) > 0 else np.zeros((1, NUM_FEATURES), dtype=np.float32)
        padding = np.repeat(last_frame, padding_needed, axis=0)
        sampled = np.vstack([all_landmarks, padding])
    return normalize_landmarks(sampled)

print('Preprocessing functions defined!')

### Run Preprocessing

In [ ]:
print('='*60)
print('STEP 1: DATA PREPROCESSING (Dual-Hand)')
print('='*60)

X, y = [], []
hands = initialize_mediapipe()

gesture_dirs = sorted([d for d in os.listdir(DATASET_PATH)
                       if os.path.isdir(os.path.join(DATASET_PATH, d))
                       and len(os.listdir(os.path.join(DATASET_PATH, d))) > 0])

print(f'Found {len(gesture_dirs)} gesture classes')
print(f'Features per frame: {NUM_FEATURES} (2 hands x 21 landmarks x 3 coords)')

for gesture_name in gesture_dirs:
    gesture_path = os.path.join(DATASET_PATH, gesture_name)
    video_files = sorted([f for f in os.listdir(gesture_path) if f.endswith(('.mp4','.avi','.mov','.mkv'))])
    print(f'\nProcessing: {gesture_name} - {len(video_files)} videos')
    for vf in video_files:
        vpath = os.path.join(gesture_path, vf)
        landmarks = extract_landmarks_from_video(vpath, hands)
        if landmarks is not None:
            X.append(landmarks)
            y.append(gesture_name)

hands.close()
X = np.array(X, dtype=np.float32)
y = np.array(y)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=TEST_SPLIT, random_state=RANDOM_SEED)

# Save
np.save(os.path.join(PREPROCESSED_DATA_PATH, 'X_train.npy'), X_train)
np.save(os.path.join(PREPROCESSED_DATA_PATH, 'X_test.npy'), X_test)
np.save(os.path.join(PREPROCESSED_DATA_PATH, 'y_train.npy'), y_train)
np.save(os.path.join(PREPROCESSED_DATA_PATH, 'y_test.npy'), y_test)
with open(os.path.join(PREPROCESSED_DATA_PATH, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(label_encoder, f)
with open(os.path.join(PREPROCESSED_DATA_PATH, 'class_names.txt'), 'w') as f:
    for cls in label_encoder.classes_: f.write(f'{cls}\n')

NUM_CLASSES = len(label_encoder.classes_)
print(f'\nTotal samples: {len(X)}, X shape: {X.shape}')
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Classes: {NUM_CLASSES}')
print('Preprocessing DONE!')

## Step 2: Data Augmentation
Generate augmented training data with noise, rotation, scaling, flipping, etc.

In [ ]:
np.random.seed(RANDOM_SEED)

def add_gaussian_noise(lm, std=NOISE_STD):
    noise = np.random.normal(0, std, lm.shape).astype(np.float32)
    aug = lm + noise
    zero_mask = np.all(lm == 0, axis=1)
    aug[zero_mask] = 0
    for i in range(len(aug)):
        for h in range(NUM_HANDS):
            s, e = h*FEATURES_PER_HAND, (h+1)*FEATURES_PER_HAND
            if np.all(lm[i, s:e] == 0): aug[i, s:e] = 0
    return aug

def random_scaling(lm, sr=SCALE_RANGE):
    scale = np.random.uniform(sr[0], sr[1])
    aug = lm.copy()
    for i in range(len(aug)):
        if np.all(aug[i]==0): continue
        for h in range(NUM_HANDS):
            s, e = h*FEATURES_PER_HAND, (h+1)*FEATURES_PER_HAND
            if np.all(aug[i,s:e]==0): continue
            f = aug[i,s:e].reshape(LANDMARKS_PER_HAND, NUM_COORDS)
            cx, cy = f[:,0].mean(), f[:,1].mean()
            f[:,0] = (f[:,0]-cx)*scale+cx
            f[:,1] = (f[:,1]-cy)*scale+cy
            aug[i,s:e] = f.flatten()
    return aug

def random_shift(lm, sr=SHIFT_RANGE):
    sx, sy = np.random.uniform(-sr,sr), np.random.uniform(-sr,sr)
    aug = lm.copy()
    for i in range(len(aug)):
        if np.all(aug[i]==0): continue
        for h in range(NUM_HANDS):
            s, e = h*FEATURES_PER_HAND, (h+1)*FEATURES_PER_HAND
            if np.all(aug[i,s:e]==0): continue
            f = aug[i,s:e].reshape(LANDMARKS_PER_HAND, NUM_COORDS)
            f[:,0]+=sx; f[:,1]+=sy
            aug[i,s:e] = f.flatten()
    return aug

def random_rotation_2d(lm, ar=ROTATION_ANGLE_RANGE):
    angle = np.radians(np.random.uniform(ar[0], ar[1]))
    cos_a, sin_a = np.cos(angle), np.sin(angle)
    aug = lm.copy()
    for i in range(len(aug)):
        if np.all(aug[i]==0): continue
        for h in range(NUM_HANDS):
            s, e = h*FEATURES_PER_HAND, (h+1)*FEATURES_PER_HAND
            if np.all(aug[i,s:e]==0): continue
            f = aug[i,s:e].reshape(LANDMARKS_PER_HAND, NUM_COORDS)
            cx, cy = f[:,0].mean(), f[:,1].mean()
            xc, yc = f[:,0]-cx, f[:,1]-cy
            f[:,0] = xc*cos_a - yc*sin_a + cx
            f[:,1] = xc*sin_a + yc*cos_a + cy
            aug[i,s:e] = f.flatten()
    return aug

def temporal_jitter(lm, ms=TEMPORAL_SHIFT):
    shift = np.random.randint(-ms, ms+1)
    aug = np.roll(lm, shift, axis=0)
    if shift > 0: aug[:shift] = 0
    elif shift < 0: aug[shift:] = 0
    return aug

def speed_variation(lm):
    sf = np.random.uniform(0.8, 1.2)
    sl = len(lm)
    nl = int(sl * sf)
    if nl < 2: return lm.copy()
    oi, ni = np.arange(sl), np.linspace(0, sl-1, nl)
    aug = np.zeros((nl, lm.shape[1]), dtype=np.float32)
    for j in range(lm.shape[1]): aug[:,j] = np.interp(ni, oi, lm[:,j])
    if nl >= SEQUENCE_LENGTH:
        idx = np.linspace(0, nl-1, SEQUENCE_LENGTH, dtype=int)
        aug = aug[idx]
    else:
        pad = np.zeros((SEQUENCE_LENGTH-nl, lm.shape[1]), dtype=np.float32)
        aug = np.vstack([aug, pad])
    return aug

def horizontal_flip(lm):
    aug = lm.copy()
    for i in range(len(aug)):
        if np.all(aug[i]==0): continue
        left = aug[i, :FEATURES_PER_HAND].copy()
        right = aug[i, FEATURES_PER_HAND:].copy()
        aug[i, :FEATURES_PER_HAND] = right
        aug[i, FEATURES_PER_HAND:] = left
        for h in range(NUM_HANDS):
            s, e = h*FEATURES_PER_HAND, (h+1)*FEATURES_PER_HAND
            if np.all(aug[i,s:e]==0): continue
            f = aug[i,s:e].reshape(LANDMARKS_PER_HAND, NUM_COORDS)
            f[:,0] = 1.0 - f[:,0]
            aug[i,s:e] = f.flatten()
    return aug

def random_finger_dropout(lm, dr=0.1):
    aug = lm.copy()
    for i in range(len(aug)):
        if np.all(aug[i]==0): continue
        for h in range(NUM_HANDS):
            s, e = h*FEATURES_PER_HAND, (h+1)*FEATURES_PER_HAND
            if np.all(aug[i,s:e]==0): continue
            f = aug[i,s:e].reshape(LANDMARKS_PER_HAND, NUM_COORDS)
            mask = np.random.random(LANDMARKS_PER_HAND) < dr
            mask[0] = False
            f[mask] = 0
            aug[i,s:e] = f.flatten()
    return aug

def augment_single_sample(lm):
    aug = lm.copy()
    augmentations = [(add_gaussian_noise,0.8),(random_scaling,0.7),(random_shift,0.6),
                     (random_rotation_2d,0.6),(temporal_jitter,0.4),(speed_variation,0.4),
                     (horizontal_flip,0.3),(random_finger_dropout,0.2)]
    for func, prob in augmentations:
        if np.random.random() < prob: aug = func(aug)
    return aug

print('Augmentation functions defined!')

### Run Augmentation

In [ ]:
print('='*60)
print('STEP 2: DATA AUGMENTATION (Dual-Hand)')
print('='*60)

X_train = np.load(os.path.join(PREPROCESSED_DATA_PATH, 'X_train.npy'))
y_train = np.load(os.path.join(PREPROCESSED_DATA_PATH, 'y_train.npy'))
print(f'Original training: X={X_train.shape}, y={y_train.shape}')

class_counts = Counter(y_train)
max_count = max(class_counts.values())

X_aug = list(X_train.copy())
y_aug = list(y_train.copy())

for cl in sorted(class_counts.keys()):
    mask = y_train == cl
    samples = X_train[mask]
    cc = class_counts[cl]
    bf = max(1, int(max_count / cc))
    ef = min(AUGMENTATION_FACTOR * bf, AUGMENTATION_FACTOR * 3)
    cnt = 0
    for s in samples:
        for _ in range(ef):
            X_aug.append(augment_single_sample(s))
            y_aug.append(cl)
            cnt += 1
    print(f'  Class {cl:>3d}: {cc} originals -> +{cnt} augmented (factor={ef})')

X_aug = np.array(X_aug, dtype=np.float32)
y_aug = np.array(y_aug)
shuffle_idx = np.random.permutation(len(X_aug))
X_aug, y_aug = X_aug[shuffle_idx], y_aug[shuffle_idx]

np.save(os.path.join(AUGMENTED_DATA_PATH, 'X_train_augmented.npy'), X_aug)
np.save(os.path.join(AUGMENTED_DATA_PATH, 'y_train_augmented.npy'), y_aug)

print(f'\nAugmented: {X_aug.shape} ({len(X_aug)/len(X_train):.1f}x increase)')
print('Augmentation DONE!')

## Step 3: Model Architecture & Training
CNN-LSTM architecture: TimeDistributed Conv1D + Plain LSTM

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, Dense, Dropout, LSTM,
    BatchNormalization, GlobalAveragePooling1D, TimeDistributed, Reshape)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (EarlyStopping, ModelCheckpoint,
    ReduceLROnPlateau, CSVLogger)

def build_gesture_model(num_classes):
    inputs = Input(shape=(SEQUENCE_LENGTH, NUM_FEATURES), name='landmark_input')
    x = TimeDistributed(Reshape((NUM_LANDMARKS, NUM_COORDS)), name='reshape')(inputs)
    # Conv1D blocks
    x = TimeDistributed(Conv1D(CNN_FILTERS[0], 3, activation='relu', padding='same'), name='conv1')(x)
    x = TimeDistributed(BatchNormalization(), name='bn1')(x)
    x = TimeDistributed(Conv1D(CNN_FILTERS[1], 3, activation='relu', padding='same'), name='conv2')(x)
    x = TimeDistributed(BatchNormalization(), name='bn2')(x)
    x = TimeDistributed(GlobalAveragePooling1D(), name='pool')(x)
    # LSTM
    x = LSTM(LSTM_UNITS[0], return_sequences=False, name='lstm1')(x)
    x = Dropout(0.4, name='drop_lstm1')(x)
    # Classification head
    x = Dense(DENSE_UNITS, activation='relu', name='dense1')(x)
    x = Dropout(DROPOUT_RATE, name='drop1')(x)
    outputs = Dense(num_classes, activation='softmax', name='output')(x)
    model = Model(inputs=inputs, outputs=outputs, name='GestureCNN_LSTM')
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

print('Model architecture defined!')

### Run Training

In [ ]:
print('='*60)
print('STEP 3: MODEL TRAINING')
print('='*60)

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Load data
X_train = np.load(os.path.join(AUGMENTED_DATA_PATH, 'X_train_augmented.npy'))
y_train = np.load(os.path.join(AUGMENTED_DATA_PATH, 'y_train_augmented.npy'))
X_test = np.load(os.path.join(PREPROCESSED_DATA_PATH, 'X_test.npy'))
y_test = np.load(os.path.join(PREPROCESSED_DATA_PATH, 'y_test.npy'))
print(f'Training: X={X_train.shape}, y={y_train.shape}')
print(f'Test: X={X_test.shape}, y={y_test.shape}')

num_classes = len(np.unique(np.concatenate([y_train, y_test])))
print(f'Classes: {num_classes}')

# Build model
model = build_gesture_model(num_classes)
model.summary()

# Class weights
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))

# Callbacks - use .keras format (TF 2.16+ default) and .h5 for backward compat
callbacks = [
    ModelCheckpoint(os.path.join(MODEL_SAVE_PATH, 'gesture_model.keras'),
                    monitor='val_accuracy', mode='max', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1),
    CSVLogger(os.path.join(MODEL_SAVE_PATH, 'training_log.csv'))
]

# TRAIN
history = model.fit(X_train, y_train, batch_size=BATCH_SIZE, epochs=EPOCHS,
                    validation_split=VALIDATION_SPLIT, class_weight=class_weight_dict,
                    callbacks=callbacks, verbose=1, shuffle=True)

# Save in both .keras (new) and .h5 (legacy) formats
model.save(os.path.join(MODEL_SAVE_PATH, 'gesture_model_final.keras'))
model.save(os.path.join(MODEL_SAVE_PATH, 'gesture_model.h5'))
model.save(os.path.join(MODEL_SAVE_PATH, 'gesture_model_final.h5'))
history_dict = {k: [float(v) for v in vals] for k, vals in history.history.items()}
with open(os.path.join(MODEL_SAVE_PATH, 'training_history.json'), 'w') as f:
    json.dump(history_dict, f, indent=2)

print(f'\nBest val accuracy: {max(history.history["val_accuracy"]):.4f}')
print('Training DONE!')

## Step 4: Model Evaluation
Generate confusion matrix, per-class accuracy, training curves, and more.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, precision_recall_fscore_support)
from tensorflow.keras.models import load_model

print('='*60)
print('STEP 4: MODEL EVALUATION')
print('='*60)

# Load data
X_test = np.load(os.path.join(PREPROCESSED_DATA_PATH, 'X_test.npy'))
y_test = np.load(os.path.join(PREPROCESSED_DATA_PATH, 'y_test.npy'))
with open(os.path.join(PREPROCESSED_DATA_PATH, 'label_encoder.pkl'), 'rb') as f:
    label_encoder = pickle.load(f)
# Load best model (try .keras first, then .h5)
model_path = os.path.join(MODEL_SAVE_PATH, 'gesture_model.keras')
if not os.path.exists(model_path):
    model_path = os.path.join(MODEL_SAVE_PATH, 'gesture_model.h5')
if not os.path.exists(model_path):
    model_path = os.path.join(MODEL_SAVE_PATH, 'gesture_model_final.keras')
print(f'Loading model from: {model_path}')
model = load_model(model_path)
with open(os.path.join(MODEL_SAVE_PATH, 'training_history.json'), 'r') as f:
    history = json.load(f)
class_names = list(label_encoder.classes_)

# Predictions
y_pred_probs = model.predict(X_test, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
overall_acc = accuracy_score(y_test, y_pred)
print(f'Overall Accuracy: {overall_acc:.4f} ({overall_acc*100:.2f}%)')

# Classification report
report = classification_report(y_test, y_pred, target_names=class_names, zero_division=0, digits=4)
with open(os.path.join(EVALUATION_PATH, 'classification_report.txt'), 'w') as f:
    f.write(report)
print(report)

# Summary metrics
metrics = {
    'overall_accuracy': float(overall_acc),
    'weighted_f1': float(f1_score(y_test, y_pred, average='weighted', zero_division=0)),
    'macro_f1': float(f1_score(y_test, y_pred, average='macro', zero_division=0)),
    'total_test_samples': int(len(y_test)),
    'num_classes': int(len(np.unique(y_test)))
}
with open(os.path.join(EVALUATION_PATH, 'summary_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Weighted F1: {metrics["weighted_f1"]:.4f}, Macro F1: {metrics["macro_f1"]:.4f}')

### Generate Evaluation Plots

In [ ]:
# ---- Training Curves ----
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
epochs_range = range(1, len(history['accuracy'])+1)
axes[0].plot(epochs_range, history['accuracy'], 'b-o', label='Train Acc', markersize=3, linewidth=2)
axes[0].plot(epochs_range, history['val_accuracy'], 'r-s', label='Val Acc', markersize=3, linewidth=2)
axes[0].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_ylim([0, 1.05])
best_ep = np.argmax(history['val_accuracy'])+1
best_acc = max(history['val_accuracy'])
axes[0].axvline(x=best_ep, color='green', linestyle='--', alpha=0.5)
axes[1].plot(epochs_range, history['loss'], 'b-o', label='Train Loss', markersize=3, linewidth=2)
axes[1].plot(epochs_range, history['val_loss'], 'r-s', label='Val Loss', markersize=3, linewidth=2)
axes[1].set_title('Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('CNN-LSTM Training History', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(EVALUATION_PATH, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Best val accuracy: {best_acc:.4f} at epoch {best_ep}')

# ---- Confusion Matrix ----
cm = confusion_matrix(y_test, y_pred)
fig_sz = max(16, len(class_names)*0.4)
plt.figure(figsize=(fig_sz, fig_sz*0.85))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names,
            yticklabels=class_names, linewidths=0.5, square=True)
plt.xlabel('Predicted', fontsize=14, fontweight='bold')
plt.ylabel('True', fontsize=14, fontweight='bold')
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.xticks(rotation=90, fontsize=8); plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(EVALUATION_PATH, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

# ---- Per-Class Accuracy ----
per_class_acc = np.nan_to_num(cm.diagonal() / cm.sum(axis=1), nan=0.0)
sorted_idx = np.argsort(per_class_acc)
sorted_names = [class_names[i] for i in sorted_idx]
sorted_acc = per_class_acc[sorted_idx]
colors = ['#e74c3c' if a<0.5 else '#f39c12' if a<0.75 else '#2ecc71' for a in sorted_acc]
fig_h = max(10, len(class_names)*0.35)
plt.figure(figsize=(12, fig_h))
bars = plt.barh(range(len(sorted_names)), sorted_acc, color=colors, edgecolor='white')
plt.yticks(range(len(sorted_names)), sorted_names, fontsize=9)
plt.xlabel('Accuracy', fontsize=12, fontweight='bold')
plt.title('Per-Class Accuracy', fontsize=14, fontweight='bold')
plt.xlim([0, 1.1]); plt.grid(axis='x', alpha=0.3)
for i, (bar, acc) in enumerate(zip(bars, sorted_acc)):
    plt.text(acc+0.02, i, f'{acc:.2f}', va='center', fontsize=8, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(EVALUATION_PATH, 'per_class_accuracy.png'), dpi=150, bbox_inches='tight')
plt.show()

# ---- Per-Class F1 ----
prec, rec, f1, sup = precision_recall_fscore_support(y_test, y_pred, labels=range(len(class_names)), zero_division=0)
sorted_idx = np.argsort(f1)
sorted_names_f1 = [class_names[i] for i in sorted_idx]
fig_h = max(10, len(class_names)*0.4)
fig, ax = plt.subplots(figsize=(14, fig_h))
y_pos = np.arange(len(sorted_names_f1))
bw = 0.25
ax.barh(y_pos-bw, prec[sorted_idx], bw, label='Precision', color='#3498db', alpha=0.8)
ax.barh(y_pos, f1[sorted_idx], bw, label='F1-Score', color='#2ecc71', alpha=0.8)
ax.barh(y_pos+bw, rec[sorted_idx], bw, label='Recall', color='#e74c3c', alpha=0.8)
ax.set_yticks(y_pos); ax.set_yticklabels(sorted_names_f1, fontsize=9)
ax.set_xlabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Per-Class Precision/Recall/F1', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower right'); ax.set_xlim([0, 1.15]); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(EVALUATION_PATH, 'per_class_f1_scores.png'), dpi=150, bbox_inches='tight')
plt.show()

# ---- Prediction Confidence ----
max_probs = np.max(y_pred_probs, axis=1)
correct = y_pred == y_test
plt.figure(figsize=(10, 6))
plt.hist(max_probs[correct], bins=30, alpha=0.7, label='Correct', color='#2ecc71', edgecolor='white')
plt.hist(max_probs[~correct], bins=30, alpha=0.7, label='Incorrect', color='#e74c3c', edgecolor='white')
plt.xlabel('Confidence', fontsize=12, fontweight='bold')
plt.ylabel('Count', fontsize=12, fontweight='bold')
plt.title('Prediction Confidence Distribution', fontsize=14, fontweight='bold')
plt.legend(fontsize=11); plt.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(EVALUATION_PATH, 'prediction_confidence.png'), dpi=150, bbox_inches='tight')
plt.show()

print('All evaluation plots saved!')

## Step 5: Save to Google Drive & Download
Copy trained model and results to Google Drive for persistence.

In [ ]:
import shutil

drive_models = os.path.join(DRIVE_SAVE_PATH, 'models')
drive_eval = os.path.join(DRIVE_SAVE_PATH, 'evaluation_results')
drive_preproc = os.path.join(DRIVE_SAVE_PATH, 'preprocessed_data')

for d in [drive_models, drive_eval, drive_preproc]:
    os.makedirs(d, exist_ok=True)

# Copy models
for f in os.listdir(MODEL_SAVE_PATH):
    src = os.path.join(MODEL_SAVE_PATH, f)
    dst = os.path.join(drive_models, f)
    shutil.copy2(src, dst)
    print(f'Copied: {f} ({os.path.getsize(src)/1024/1024:.2f} MB)')

# Copy evaluation results
for f in os.listdir(EVALUATION_PATH):
    shutil.copy2(os.path.join(EVALUATION_PATH, f), os.path.join(drive_eval, f))
    print(f'Copied: {f}')

# Copy preprocessed data (label encoder + class names)
for f in ['label_encoder.pkl', 'class_names.txt']:
    src = os.path.join(PREPROCESSED_DATA_PATH, f)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(drive_preproc, f))
        print(f'Copied: {f}')

print('\n' + '='*60)
print('ALL FILES SAVED TO GOOGLE DRIVE!')
print('='*60)
print(f'Models:     {drive_models}')
print(f'Evaluation: {drive_eval}')
print(f'Data:       {drive_preproc}')
print('\nDownload gesture_model.h5 and label_encoder.pkl to your local machine.')
print('Then run: streamlit run app.py')